# 多 Agent 協作系統 - 研究助手團隊

## 專案概述

本範例展示如何構建一個多 Agent 協作系統，模擬一個研究團隊的工作流程：

- **研究員 (Researcher)**: 負責搜尋和收集資訊
- **分析師 (Analyst)**: 負責分析資料並提取關鍵見解
- **撰寫者 (Writer)**: 負責撰寫最終報告
- **審查者 (Reviewer)**: 負責審查並確保品質

## 學習目標

- 理解多 Agent 架構設計
- 掌握 Agent 之間的通信機制
- 學習任務委派與協調
- 實現工作流程編排

## 難度: ⭐⭐⭐⭐⭐
## 預計時間: 2-3小時

## 1. 環境設置與導入

In [ ]:
# 安裝必要套件
!pip install langchain langgraph langchain-openai tavily-python python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import json

# 載入環境變數
load_dotenv()

# 檢查 API Keys
assert os.getenv("OPENAI_API_KEY"), "請設置 OPENAI_API_KEY"
assert os.getenv("TAVILY_API_KEY"), "請設置 TAVILY_API_KEY"

print("✅ 環境設置完成")

## 2. 定義共享狀態

所有 Agent 共享的狀態結構：

In [ ]:
class ResearchState(TypedDict):
    """研究團隊的共享狀態"""
    
    # 主要任務
    topic: str  # 研究主題
    messages: Annotated[List, add_messages]  # 消息歷史
    
    # 各階段產出
    raw_research: str  # 原始研究資料
    analysis: str  # 分析結果
    draft_report: str  # 初稿
    final_report: str  # 最終報告
    
    # 控制流程
    current_step: str  # 當前步驟
    review_feedback: str  # 審查反饋
    revision_count: int  # 修訂次數

## 3. 定義工具

為 Researcher Agent 提供搜尋工具：

In [ ]:
# 初始化搜尋工具
search_tool = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=False
)

tools = [search_tool]

# 測試搜尋工具
test_results = search_tool.invoke("LangChain LangGraph")
print(f"✅ 搜尋工具正常 (找到 {len(test_results)} 個結果)")

## 4. 定義各個 Agent

### 4.1 研究員 Agent (Researcher)

In [ ]:
researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位專業的研究員。你的任務是：
    
1. 針對給定主題進行深入研究
2. 使用搜尋工具收集相關資訊
3. 整理並組織研究發現
4. 確保資訊來源可靠且時效性強

研究主題: {topic}

請進行全面的研究，涵蓋以下方面：
- 基本概念和定義
- 最新發展和趨勢
- 實際應用案例
- 潛在挑戰和限制

使用搜尋工具收集資訊，然後提供結構化的研究摘要。"""),
    MessagesPlaceholder(variable_name="messages"),
])

researcher_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)
researcher_agent = researcher_prompt | researcher_llm

def researcher_node(state: ResearchState) -> dict:
    """研究員節點"""
    response = researcher_agent.invoke({
        "topic": state["topic"],
        "messages": state["messages"]
    })
    
    return {
        "messages": [response],
        "current_step": "research"
    }

print("✅ 研究員 Agent 定義完成")

### 4.2 分析師 Agent (Analyst)

In [ ]:
analyst_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位資深數據分析師。你的任務是：

1. 仔細審查研究員收集的資料
2. 提取關鍵見解和模式
3. 進行深入分析
4. 識別重要趨勢和發現

研究主題: {topic}

請提供：
- 核心發現 (3-5點)
- 趨勢分析
- 關鍵見解
- 建議的重點方向

以結構化和專業的方式呈現你的分析。"""),
    MessagesPlaceholder(variable_name="messages"),
])

analyst_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
analyst_agent = analyst_prompt | analyst_llm

def analyst_node(state: ResearchState) -> dict:
    """分析師節點"""
    response = analyst_agent.invoke({
        "topic": state["topic"],
        "messages": state["messages"]
    })
    
    return {
        "messages": [response],
        "analysis": response.content,
        "current_step": "analysis"
    }

print("✅ 分析師 Agent 定義完成")

### 4.3 撰寫者 Agent (Writer)

In [ ]:
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位專業的技術文檔撰寫者。你的任務是：

1. 基於研究和分析結果撰寫清晰的報告
2. 使用專業但易懂的語言
3. 組織良好的結構
4. 包含適當的細節和例子

研究主題: {topic}

報告應包含：
## 摘要
## 引言
## 核心發現
## 詳細分析
## 應用場景
## 挑戰與限制
## 結論與建議

撰寫一份專業、全面且易於理解的研究報告。"""),
    MessagesPlaceholder(variable_name="messages"),
])

writer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
writer_agent = writer_prompt | writer_llm

def writer_node(state: ResearchState) -> dict:
    """撰寫者節點"""
    response = writer_agent.invoke({
        "topic": state["topic"],
        "messages": state["messages"]
    })
    
    return {
        "messages": [response],
        "draft_report": response.content,
        "current_step": "writing"
    }

print("✅ 撰寫者 Agent 定義完成")

### 4.4 審查者 Agent (Reviewer)

In [ ]:
reviewer_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一位嚴格的品質審查者。你的任務是：

1. 審查報告的品質和完整性
2. 檢查邏輯連貫性
3. 驗證資訊準確性
4. 評估是否達到專業標準

研究主題: {topic}

審查標準：
- 內容完整性 (是否涵蓋所有重要方面)
- 邏輯清晰度 (結構是否合理)
- 專業性 (語言和表達)
- 準確性 (資訊是否可靠)

請提供：
1. 整體評分 (1-10)
2. 優點
3. 需要改進的地方
4. 是否批准發布 (APPROVED/NEEDS_REVISION)

以 JSON 格式回覆：
{{
  "score": 評分,
  "strengths": ["優點1", "優點2"],
  "improvements": ["改進點1", "改進點2"],
  "status": "APPROVED" 或 "NEEDS_REVISION",
  "feedback": "詳細反饋"
}}"""),
    MessagesPlaceholder(variable_name="messages"),
])

reviewer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
reviewer_agent = reviewer_prompt | reviewer_llm

def reviewer_node(state: ResearchState) -> dict:
    """審查者節點"""
    response = reviewer_agent.invoke({
        "topic": state["topic"],
        "messages": state["messages"]
    })
    
    # 嘗試解析 JSON 反饋
    try:
        feedback = json.loads(response.content)
    except:
        # 如果解析失敗，創建基本反饋
        feedback = {
            "status": "APPROVED" if "APPROVED" in response.content else "NEEDS_REVISION",
            "feedback": response.content
        }
    
    return {
        "messages": [response],
        "review_feedback": json.dumps(feedback, ensure_ascii=False),
        "current_step": "review"
    }

print("✅ 審查者 Agent 定義完成")

## 5. 構建工作流程圖

使用 LangGraph 編排多 Agent 工作流：

In [ ]:
# 工具執行節點
tool_node = ToolNode(tools)

# 決策函數：研究員是否需要調用工具
def should_use_tools(state: ResearchState) -> Literal["tools", "analyst"]:
    """判斷是否需要使用工具"""
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "analyst"

# 決策函數：審查結果
def should_revise(state: ResearchState) -> Literal["writer", "end"]:
    """判斷是否需要修訂"""
    try:
        feedback = json.loads(state.get("review_feedback", "{}"))
        status = feedback.get("status", "APPROVED")
        revision_count = state.get("revision_count", 0)
        
        # 最多修訂 2 次
        if status == "NEEDS_REVISION" and revision_count < 2:
            return "writer"
    except:
        pass
    
    return "end"

# 修訂計數節點
def increment_revision(state: ResearchState) -> dict:
    """增加修訂計數"""
    return {"revision_count": state.get("revision_count", 0) + 1}

# 完成節點
def finalize_report(state: ResearchState) -> dict:
    """完成報告"""
    return {
        "final_report": state.get("draft_report", ""),
        "current_step": "completed"
    }

# 構建狀態圖
workflow = StateGraph(ResearchState)

# 添加節點
workflow.add_node("researcher", researcher_node)
workflow.add_node("tools", tool_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("writer", writer_node)
workflow.add_node("reviewer", reviewer_node)
workflow.add_node("increment", increment_revision)
workflow.add_node("finalize", finalize_report)

# 設置邊
workflow.add_edge(START, "researcher")
workflow.add_conditional_edges("researcher", should_use_tools)
workflow.add_edge("tools", "researcher")
workflow.add_edge("analyst", "writer")
workflow.add_edge("writer", "reviewer")
workflow.add_conditional_edges(
    "reviewer",
    should_revise,
    {
        "writer": "increment",
        "end": "finalize"
    }
)
workflow.add_edge("increment", "writer")
workflow.add_edge("finalize", END)

# 編譯圖
app = workflow.compile()

print("✅ 工作流程圖構建完成")

## 6. 可視化工作流程

In [ ]:
try:
    from IPython.display import Image, display
    
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"無法顯示圖形: {e}")
    print("\n工作流程描述:")
    print("START → Researcher → [Tools?] → Analyst → Writer → Reviewer → [Approved?] → END")

## 7. 執行研究任務

In [ ]:
# 定義研究主題
research_topic = "LangGraph 在構建 AI Agent 系統中的應用"

# 初始化狀態
initial_state = {
    "topic": research_topic,
    "messages": [HumanMessage(content=f"請研究: {research_topic}")],
    "raw_research": "",
    "analysis": "",
    "draft_report": "",
    "final_report": "",
    "current_step": "init",
    "review_feedback": "",
    "revision_count": 0
}

print(f"🚀 開始研究: {research_topic}")
print("=" * 80)

In [ ]:
# 執行工作流程（流式輸出）
for step, output in enumerate(app.stream(initial_state), 1):
    print(f"\n{'='*80}")
    print(f"步驟 {step}: {list(output.keys())[0].upper()}")
    print(f"{'='*80}")
    
    node_name = list(output.keys())[0]
    node_output = output[node_name]
    
    # 顯示當前步驟的關鍵資訊
    if "current_step" in node_output:
        print(f"當前階段: {node_output['current_step']}")
    
    if "messages" in node_output and node_output["messages"]:
        last_msg = node_output["messages"][-1]
        if hasattr(last_msg, "content"):
            content = last_msg.content
            # 限制輸出長度
            if len(content) > 500:
                print(f"輸出: {content[:500]}...")
            else:
                print(f"輸出: {content}")

print("\n" + "=" * 80)
print("✅ 研究任務完成")
print("=" * 80)

## 8. 查看最終報告

In [ ]:
# 獲取最終狀態
final_state = app.invoke(initial_state)

print("📊 最終研究報告")
print("=" * 80)
print(final_state.get("final_report", "報告生成失敗"))
print("\n" + "=" * 80)

# 顯示審查反饋
if final_state.get("review_feedback"):
    print("\n📝 審查反饋")
    print("=" * 80)
    try:
        feedback = json.loads(final_state["review_feedback"])
        print(json.dumps(feedback, indent=2, ensure_ascii=False))
    except:
        print(final_state["review_feedback"])

print(f"\n修訂次數: {final_state.get('revision_count', 0)}")

## 9. 保存報告

In [ ]:
import datetime

# 生成文件名
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"research_report_{timestamp}.md"

# 保存報告
with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# 研究報告: {research_topic}\n\n")
    f.write(f"生成時間: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("---\n\n")
    f.write(final_state.get("final_report", ""))
    f.write("\n\n---\n\n")
    f.write("## 審查資訊\n\n")
    f.write(f"修訂次數: {final_state.get('revision_count', 0)}\n\n")
    
    if final_state.get("review_feedback"):
        f.write("### 審查反饋\n\n")
        f.write("```json\n")
        f.write(final_state["review_feedback"])
        f.write("\n```\n")

print(f"✅ 報告已保存至: {filename}")

## 10. 實驗與擴展

### 可以嘗試的擴展：

1. **添加更多 Agent 角色**
   - 事實核查員 (Fact Checker)
   - SEO 優化師
   - 翻譯員

2. **改進協作機制**
   - Agent 之間的直接對話
   - 並行處理某些任務
   - 動態任務分配

3. **增強工具集**
   - 資料庫查詢
   - API 調用
   - 文件處理

4. **優化品質控制**
   - 更嚴格的審查標準
   - 多輪反饋循環
   - 人工介入選項

## 總結

本範例展示了：

✅ 多 Agent 系統架構設計
✅ Agent 之間的狀態共享
✅ 工作流程編排與控制
✅ 工具集成與使用
✅ 品質控制與迭代改進

### 關鍵學習點：

1. **狀態管理**: 使用 TypedDict 定義共享狀態
2. **任務委派**: 每個 Agent 負責特定任務
3. **流程控制**: 使用條件邊實現決策邏輯
4. **工具使用**: 整合外部工具擴展能力
5. **迭代優化**: 實現反饋循環改進輸出

### 下一步：

- 嘗試不同的研究主題
- 調整各 Agent 的 Prompt
- 添加新的 Agent 角色
- 實現更複雜的工作流程